# DSAR × Lakeflow Declarative Pipelines · 01 · Pipeline (append silver + Auto Loader)

**Attach this as a Lakeflow Declarative Pipeline source** (Jobs & Pipelines →
Create pipeline → add this notebook). Do not "Run all" interactively — Databricks
executes it as the pipeline graph.

Modern API: `from pyspark import pipelines as dp` — `@dp.table` for streaming
tables, `@dp.materialized_view` for the aggregate gold. (This replaces the legacy
`import dlt` module.)

### Medallion — ingested from files by Auto Loader

```
/Volumes/.../raw_user/landing/*.json
        │  Auto Loader (cloudFiles)
        ▼
raw_user ─stream─▶ bronze_user ─stream─▶ silver_user ─batch MV─▶ gold_user
  (ingest)          mask PII             clean/validate         per-customer aggregate
```

`raw_user` is a **streaming table fed by Auto Loader** reading JSON from the UC
volume landing zone (both `initial/` and `incremental/` subfolders). Each
downstream **streaming** read uses `.option("skipChangeCommits", "true")` so that
when a DSAR erasure (notebook `02`) runs a `DELETE`/`UPDATE` on an upstream table,
the stream **skips that non-append commit instead of failing**. Skipping a commit
only advances the offset (no data read) → negligible latency.

Set in the pipeline **Configuration**: `dsar.catalog`, `dsar.schema`, and
(optionally) `dsar.volume`; set the pipeline's default catalog + target schema to
the same, so tables land next to the volume.


## 0. Config


In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

def cfg(key, default):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

CATALOG = cfg("dsar.catalog", "dkushari_uc")
SCHEMA  = cfg("dsar.schema",  "allegiant_air_sdp_dsar")
VOLUME  = cfg("dsar.volume",  "raw_user")
FQ      = f"{CATALOG}.{SCHEMA}"
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/landing"   # Auto Loader source (initial/ + incremental/)
print("Auto Loader landing:", LANDING, "| target schema:", FQ)


## 1. Raw — Auto Loader ingest from the volume (streaming)\n\nIngests the landing JSON files into a `raw_user` streaming table. This is the\nsource of truth the pipeline consumes; it still holds cleartext PII (bronze masks).


In [ ]:
# Explicit schema => deterministic, no _rescued_data drift, no schemaLocation needed.
RAW_SCHEMA = ("event_id string, user_id string, email string, full_name string, "
              "profile_json string, revenue double, event_ts string, _ingest_ts string")

@dp.table(
    name="raw_user",
    comment="Auto Loader ingest of landing JSON files (cleartext PII). Recurses landing/ so both initial/ and incremental/ files are picked up.",
    table_properties={"quality": "raw"},
)
def raw_user():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .schema(RAW_SCHEMA)
            .load(LANDING))


## 2. Bronze — mask PII inline (streaming)\n\nReads `raw_user` as a stream and applies **native-SQL** PII masking as it writes:\n`email`/`full_name` and the in-JSON `contact.*` values → `***REDACTED***`;\n`user_id`/`revenue`/`event_ts` preserved. `skipChangeCommits` on the read of\n`raw_user` keeps this stream alive when an erasure deletes from raw.


In [ ]:
REDACT = "***REDACTED***"

def _mask_json(col):
    # in-JSON masking, native SQL — quote-anchored keys so "name" != "appName"
    e = f"regexp_replace({col}, '(\"email\" *: *)\"[^\"]*\"', '$1\"{REDACT}\"')"
    e = f"regexp_replace({e}, '(\"name\" *: *)\"[^\"]*\"', '$1\"{REDACT}\"')"
    return e

@dp.table(
    name="bronze_user",
    comment="Streaming bronze: PII masked inline; user_id/revenue preserved.",
    table_properties={"quality": "bronze"},
)
def bronze_user():
    src = (spark.readStream
           .option("skipChangeCommits", "true")   # survive erasure on raw_user
           .table(f"{FQ}.raw_user"))
    return src.select(
        "event_id",
        "user_id",                                    # stable key — NOT masked
        F.lit(REDACT).alias("email"),                 # PII -> redact
        F.lit(REDACT).alias("full_name"),             # PII -> redact
        F.expr(_mask_json("profile_json")).alias("profile_json"),  # in-JSON PII -> redact
        "revenue", "event_ts", "_ingest_ts",          # non-PII -> preserve
    )


## 3. Silver — cleaned / validated events (streaming)

Reads **bronze** as a stream, standardizes and validates at event grain (drop rows
with no `user_id`, cast revenue, add a processing timestamp). Still a plain Delta
streaming table, so an erasure can `DELETE` from it directly. `skipChangeCommits`
on the read of bronze keeps this stream alive when an erasure deletes from bronze.


In [ ]:
@dp.table(
    name="silver_user",
    comment="Streaming silver: cleaned & validated events, keyed on user_id.",
    table_properties={"quality": "silver"},
)
def silver_user():
    return (spark.readStream
            .option("skipChangeCommits", "true")   # survive erasure on bronze_user
            .table(f"{FQ}.bronze_user")
            .where(F.col("user_id").isNotNull())
            .withColumn("revenue", F.col("revenue").cast("double"))
            .withColumn("_silver_ts", F.current_timestamp())
            .select("event_id", "user_id", "email", "full_name",
                    "profile_json", "revenue", "event_ts", "_silver_ts"))


## 4. Gold — per-customer aggregate (materialized view)

Reads **silver** and rolls up to one row per customer. A **materialized view** =
batch recompute (`spark.read`, not `readStream`), so it reflects erasures
automatically on refresh and holds no streaming checkpoint state that could
resurrect an erased subject. You **cannot `DELETE` from a view** — the erasure
notebook deletes the base tables and **refreshes** this MV (see `02`).


In [ ]:
@dp.materialized_view(
    name="gold_user",
    comment="Per-customer lifetime rollup, recomputed from silver each refresh.",
    table_properties={"quality": "gold"},
)
def gold_user():
    return (spark.read.table(f"{FQ}.silver_user")   # batch read => materialized view
            .groupBy("user_id")
            .agg(F.sum("revenue").alias("lifetime_revenue"),
                 F.count("*").alias("event_count"),
                 F.max("event_ts").alias("last_event_ts")))
